# GA4 Silver Layer Pipeline

**Purpose**: Transform bronze GA4 events into clean, analytics-ready silver tables with flattened nested structures.

**Architecture**:
- Process daily GA4 batch files incrementally.
- Prevents duplicate events within source data
- Split into external (customer traffic) and internal (team testing from Virginia) tables
- Data quality: invalid records (null page_location, page_title, or event_timestamp) can be filter in where clause as needed.

**Why Merge Tables?**
- **Rerun immunity**: safe to rerun without reprocessing old data
- **Incremental processing**: Only new bronze data since last run is processed


**Flow**: Bronze events → Flatten nested structures (event_params, geo, device, traffic_source) → Filter by traffic type → Deduplicate → Write to silver delta tables

### Step 1: Transformation Functions

Defines helper functions to flatten GA4's nested JSON structure:
- `flatten_params()` - Pivots event_params array into individual columns
- `flatten_struct()` - Expands structs (geo, device, traffic_source) into prefixed columns  
- `flatten_sub_struct()` - Expands nested structs within parent structs
- `flatten_items()` - Explodes items array for e-commerce events

In [0]:


# ── Imports ───────────────────────────────────────────────────────────────────
from pyspark.sql.functions import explode_outer, coalesce, col, first
from pyspark.sql.functions import explode_outer, col, coalesce, first, when
# ── Param Lists ───────────────────────────────────────────────────────────────

df= dlt.read_stream("workspace.bronze.events")
EVENT_PARAMS = [
    "term", "page_title", "ga_session_number", "source",
    "page_referrer", "page_location", "traffic_type", "session_engaged", "engagement_time_msec","percent_scrolled"
]

USER_PROPERTIES = [
    # paste your user_properties keys here
]

ITEMS_FIELDS = [
    "item_id", "item_name", "item_brand", "item_variant",


]

# ── Functions ──────────────────────────────────────────from pyspark.sql.functions import explode_outer, col, coalesce, first, when

def flatten_params(df, array_col, params):
    # Flatten GA4 key/value struct array into individual columns without pivot
    df_exploded = (
        df.withColumn("_param", explode_outer(col(array_col)))
          .withColumn("_value", coalesce(
              col("_param.value.string_value"),
              col("_param.value.int_value").cast("string"),
              col("_param.value.float_value").cast("string"),
              col("_param.value.double_value").cast("string"),
          ))
          .filter(col("_param.key").isin(params))
    )

    group_cols = [c for c in df.columns if c != array_col]

    return (
        df_exploded.groupBy(group_cols)
          .agg(*[
              first(when(col("_param.key") == p, col("_value"))).alias(p)
              for p in params
          ])
    )

def flatten_items(df, fields):
    # Flat struct array — explode_outer keeps rows with null items
    return (
        df.withColumn("_item", explode_outer(col("items")))
          .select(
              [c for c in df.columns if c != "items"] +
              [col(f"_item.{f}").alias(f) for f in fields]
          )
    )

def flatten_struct(df, struct_col, prefix=None):
    # Flatten a top-level struct into individual columns with optional prefix
    fields = [f.name for f in df.schema[struct_col].dataType.fields]
    return (
        df.select(
            [c for c in df.columns if c != struct_col] +
            [col(f"{struct_col}.{f}").alias(f"{prefix}_{f}" if prefix else f) for f in fields]
        )
    )

def flatten_sub_struct(df, parent_col, sub_col, prefix):
    # Flatten one sub-struct inside a top-level struct with a prefix
    fields = [f.name for f in df.schema[parent_col].dataType[sub_col].dataType.fields]
    return df.select(
        [c for c in df.columns] +
        [col(f"{parent_col}.{sub_col}.{f}").alias(f"{prefix}_{f}") for f in fields]
    )



In [0]:
def df_transform(df):
    df_silver = (
        df
        .transform(lambda df: flatten_params(df, "event_params", EVENT_PARAMS))
        .transform(lambda df: flatten_struct(df, "geo"))
        .transform(lambda df: flatten_struct(df, "traffic_source", prefix="traffic_source"))
        .transform(lambda df: flatten_struct(df, "collected_traffic_source"))
        .transform(lambda df: flatten_struct(df, "device"))
        .drop(
            "session_traffic_source_last_click", "privacy_info", "user_properties",
            "ecommerce", "event_params", "items", "event_dimensions", "traffic_source",
            "collected_traffic_source", "device", "web_info", "app_info",
            "event_previous_timestamp", "event_server_timestamp_offset",
            "event_bundle_sequence_id", "event_value_in_usd", "user_id",
            "user_pseudo_id", "user_first_touch_timestamp", "stream_id", "platform",
            "batch_event_index", "batch_page_id", "batch_ordering_id",
            "analytics_storage", "ads_storage", "uses_transient_token",
            "revenue", "geo", "publisher", "user_ltv"
        ))
    return df_silver

### Step 2: External Traffic Prep

`prepare_external_traffic()` function filters and prepares customer/visitor events:
- Excludes Virginia internal traffic
- Converts event_date from yyyyMMdd string → DATE type
- Creates `_row_key` as MD5 hash (timestamp + event_name + page_location + country + city)
- Drops duplicate events based on _row_key

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date

# ── External Traffic Preparation ──────────────────────────────────────────────
def prepare_external_traffic(df_silver):
    from pyspark.sql.functions import col, to_date, md5, concat_ws, trim
    # Filter, select, and deduplicate GA4 external traffic for silver layer
    df_ready = (
        df_silver
        .filter((trim(col("traffic_type")) != "internal") | col("traffic_type").isNull())
        .select(
            "event_date", "event_name", "event_timestamp", "is_active_user", "term", "page_title","ga_session_number", "source", "page_referrer", "page_location", "traffic_type", "session_engaged", "city", "country", "continent",
            "region", "sub_continent", "metro", "traffic_source_name",
            "traffic_source_medium", "manual_source", "manual_medium", "manual_term",
            "category", "mobile_brand_name", "mobile_model_name", "operating_system",
            "operating_system_version", "language", "engagement_time_msec", "percent_scrolled",
        )
        .withColumn("event_date", to_date(col("event_date").cast("string"), "yyyyMMdd"))
        .withColumn("_row_key", md5(concat_ws("|",
            col("event_timestamp").cast("string"),
            col("event_name"),
            col("page_location"),
            col("country"),
            col("city"),
        )))
      #  .dropDuplicates(["_row_key"])
    )
    return df_ready



### Step 3: External Traffic Table

`ga4_silver` - Streaming table for external customer traffic
- Reads from bronze, flattens nested structures, drops unnecessary columns
- Filters for external traffic and deduplicates via `prepare_external_traffic()`
- Expectations drop rows with null page_location, page_title, or event_timestamp

### Step 4: Internal Traffic Prep

`prepare_internal_traffic()` function filters and prepares team testing events:
- Includes ONLY Virginia internal traffic (region = Virginia AND traffic_type = internal)
- Uses same column selection, date conversion, and deduplication as external
- Separates internal monitoring traffic for separate analysis without polluting customer analytics

### Step 5: Internal Traffic Silver Table

`ga4_silver_internal` - Streaming table for internal team traffic

**Data Quality Expectations** (drop invalid records):
* `page_location` must not be null
* `page_title` must not be null
* `event_timestamp` must not be null

**Processing Flow**:
1. Read streaming data from bronze layer
2. Flatten nested structures (same transformations as external)
3. Drop unnecessary columns
4. Filter for Virginia internal traffic and deduplicate
5. Write to internal silver streaming table

In [0]:
from pyspark.sql.functions import col, to_date, md5, concat_ws, trim

def prepare_internal_traffic(df_silver):
    # Filter, select, and deduplicate GA4 internal traffic (Virginia) for silver layer
    df_read_internal = (
        df_silver
        .filter((trim(col("traffic_type")) == "internal") & 
                (col("region") == "Virginia"))
        .select(
            "event_date", "event_name", "event_timestamp", "is_active_user", "term", "page_title",
            "ga_session_number", "source", "page_referrer", "page_location",
            "traffic_type", "session_engaged", "city", "country", "continent",
            "region", "sub_continent", "metro", "traffic_source_name",
            "traffic_source_medium", "manual_source", "manual_medium", "manual_term",
            "category", "mobile_brand_name", "mobile_model_name", "operating_system",
            "operating_system_version", "language", "engagement_time_msec", "percent_scrolled",
        )
        .withColumn("event_date", to_date(col("event_date").cast("string"), "yyyyMMdd"))
        .withColumn("_row_key", md5(concat_ws("|",
            col("event_timestamp").cast("string"),
            col("event_name"),
            col("page_location"),
            col("country"),
            col("city"),
        )))
       # .dropDuplicates(["_row_key"]))
     )
    return df_read_internal



In [0]:
from delta.tables import DeltaTable
import dlt


@dlt.table(
    name="ga4_silver",
    comment="Cleaned and flattened GA4 events from bronze",
    table_properties={
        "pipelines.schemaEvolution": "addNewColumns"
    } )
#@dlt.expect_or_drop("valid_page_location", "page_location IS NOT NULL")
#@dlt.expect_or_drop("valid_page_title",    "page_title IS NOT NULL")
@dlt.expect_or_drop("valid_event_time",    "event_timestamp IS NOT NULL")
def ga4_silver():
    df= dlt.read_stream("workspace.bronze.events")
    df_silver=df_transform(df)

    return prepare_external_traffic(df_silver)



@dlt.table(
    name="ga4_silver_internal",
    comment="Cleaned and flattened GA4 internal events from bronze",
    table_properties={"pipelines.schemaEvolution": "addNewColumns"}
)
#@dlt.expect_or_drop("valid_page_location", "page_location IS NOT NULL")
#@dlt.expect_or_drop("valid_page_title",    "page_title IS NOT NULL")
@dlt.expect_or_drop("valid_event_time",    "event_timestamp IS NOT NULL")
def ga4_silver_internal():
    df = dlt.read_stream("workspace.bronze.events")
    df_silver = df_transform(df)
    return prepare_internal_traffic(df_silver)




@dlt.table(
    name="ga4_quarantine",
    comment="GA4 events with missing required fields",
    table_properties={"pipelines.schemaEvolution": "addNewColumns"}
)

def ga4_quarantine():
    df = dlt.read_stream("workspace.bronze.events")
    return df_transform(df).filter(
        col("page_location").isNull() |
        col("page_title").isNull() |
        col("event_timestamp").isNull()
    )

Step 7: route bad data to Qurantine table